# 01 — Data ingest and the true time base (stage 2)

**Purpose.** Merge the 45 monthly SCADA exports of the Ktora section into one
minute-level table, give every tag a canonical name, and put every timestamp on true UTC.

**Inputs.** `Raw data/*.CSV` (read-only; byte-identical to the stage-0 checksums).
**Outputs.** `data/processed/01_ingest/`: `scada_1min_utc.parquet`, `ingest_manifest.json`,
`ingest_qa.csv`, `tag_map.csv`.

**Rules.** No value is changed at this stage: anomalies are only counted, and cleaning belongs to
stage 5. The time rule fixes finding F-01 of the review: the original pipeline read Israel local
time as if it were UTC.

**Where it is documented.** `docs/data_processing_record.md`, OP-01 to OP-05; module
`solarpv.data.ingest`; tests T-05 and T-08.

## 1. The raw files

In [1]:
import pandas as pd

from solarpv.data import ingest

files = ingest.raw_files()
print(len(files), "files:", files[0].name, "…", files[-1].name)
print("byte-identical to the stage-0 checksums:", ingest._verify_against_checksums(files))

45 files: 2109.CSV … 2505.CSV


byte-identical to the stage-0 checksums: 45


**What it shows.** 45 monthly files, from 2109.CSV to
2505.CSV, each byte-identical to the checksums taken when the data were received
(`00_data_audit/raw_checksums.sha256`). Every run of the stage repeats this check
(45 files verified in the last run), so a changed raw
file stops the pipeline instead of silently changing the results.

In [2]:
raw = pd.read_csv(files[1], nrows=3, dtype=str)
print(raw.shape[1], "columns")
raw.iloc[:, :6]

28 columns


,t_stamp,Sites/PVS/Ktora/IEC/Mode,Sites/PVS/Ktora/IEC/SP_W,Sites/PVS/Ktora/IEC/SP_Q,Sites/PVS/Ktora/IEC/SP_V,Sites/PVS/Ktora/IEC/SP_PF
0,2021-10-01 00:00:00.000,2.0,100.0,366.4893803086984,163.68790197070263,0.9800000190734863
1,2021-10-01 00:01:00.000,2.0,100.0,367.18748711322957,163.3944367051308,0.9800000190734863
2,2021-10-01 00:02:00.000,2.0,100.0,365.5435551635419,162.92195912068132,0.9800000190734863


**What it shows.** One raw file: a local-time column `t_stamp` and 27
SCADA tag paths, one value per minute. The stamps carry no time zone: they are Israel local time,
with daylight-saving time.

## 2. Canonical names

In [3]:
pd.DataFrame({"raw tag": list(ingest.TAG_MAP),
              "name": list(ingest.TAG_MAP.values())})

,raw tag,name
0,Sites/PVS/Ktora/IEC/Mode,iec_mode
1,Sites/PVS/Ktora/IEC/SP_W,iec_sp_w
2,Sites/PVS/Ktora/IEC/SP_Q,iec_sp_q
3,Sites/PVS/Ktora/IEC/SP_V,iec_sp_v
4,Sites/PVS/Ktora/IEC/SP_PF,iec_sp_pf
5,Sites/PVS/Ktora/meteo/amb_temp/st02_amb_temp/eu,ambient_temp
6,Sites/PVS/Ktora/meteo/irrad/st01_irrad/eu,irradiance
7,Sites/PVS/Ktora/meteo/mod_temp/st01_mod_temp1/eu,mod_temp1
8,Sites/PVS/Ktora/meteo/mod_temp/st01_mod_temp2/eu,mod_temp2
9,Sites/PVS/Ktora/meteo/precipitation/st05_preci...,precipitation


**What it shows.** The 27 tags and their names in the rebuild (PROJECT_MASTER.md §6.1).

**What it means.** `irradiance` is the in-plane (POA) sensor, not a horizontal one, and
`pyranometer` is an identical second in-plane sensor kept for quality checks (F-13, D-09). The
site has no horizontal GHI measurement, which matters for the definition of the clear-sky
reference in stage 6.

## 3. Daylight-saving time: the repeated October hour and the missing March hour

In [4]:
def local_stamps(name):
    path = next(p for p in files if p.name == name)
    s = pd.read_csv(path, usecols=["t_stamp"], dtype=str)["t_stamp"]
    return pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S.%f")

october = local_stamps("2110.CSV")
repeated = october[october.duplicated(keep=False)]
print("October 2021: local stamps that occur twice:", repeated.size,
      "| from", repeated.min(), "to", repeated.max())
utc = ingest.to_utc(october)
print("after conversion: duplicates", utc.duplicated().sum(),
      "| the two passes of 01:00 local ->", utc[october == repeated.min()].tolist())

march = local_stamps("2203.CSV")
gap = pd.date_range("2022-03-25 02:00", "2022-03-25 02:59", freq="min")
print("March 2022: local stamps between 02:00 and 02:59 on 25 March:", march.isin(gap).sum())

October 2021: local stamps that occur twice: 120 | from 2021-10-31 01:00:00 to 2021-10-31 01:59:00
after conversion: duplicates 0 | the two passes of 01:00 local -> [Timestamp('2021-10-30 22:00:00+0000', tz='UTC'), Timestamp('2021-10-30 23:00:00+0000', tz='UTC')]


March 2022: local stamps between 02:00 and 02:59 on 25 March: 0


**What it shows.** When the clocks go back in October, local 01:00–01:59 occurs twice: the
file holds both passes, one after the other. When the clocks go forward in March, local 02:00–02:59
does not exist and the file has no rows for it.

**What it means.** The rebuild localises the stamps to `Asia/Jerusalem` (the repeated hour is
resolved by file order) and converts them to UTC; afterwards every minute is unique. **In the
original** (notebook 03) the local stamps were simply labelled UTC, so the whole data set was
shifted by 2 hours in winter and 3 in summer against the sun (F-01); the repeated October hour
kept two values under one label, and the March hour stayed missing (F-10).

In [5]:
from solarpv.config.paths import PROCESSED_DIR

STAGE = PROCESSED_DIR / "01_ingest"
minutes = pd.read_parquet(STAGE / "scada_1min_utc.parquet", columns=["s01_inv1_AC_kW"])
ingest.check_time_axis(minutes.index)
print(len(minutes), "minutes |", minutes.index.min(), "->", minutes.index.max())

1971360 minutes | 2021-08-31 21:00:00+00:00 -> 2025-05-31 20:59:00+00:00


**What it shows.** The merged table passes the time-axis check: 1,971,360 minutes,
from 2021-08-31 21:00:00+00:00 to 2025-05-31 20:59:00+00:00 UTC, with no duplicate and no missing minute.

**Rule used** (manifest): `tz_localize('Asia/Jerusalem', ambiguous='infer', nonexistent='raise') -> tz_convert('UTC')`.

## 4. What the raw data contain (counted, not changed)

In [6]:
qa = pd.read_csv(STAGE / "ingest_qa.csv")
qa[["column", "min", "p50", "max", "n_negative", "n_above_inverter_flag"]].round(1)

,column,min,p50,max,n_negative,n_above_inverter_flag
0,iec_mode,0.0,2.0,2.0,0,0
1,iec_sp_w,0.0,100.0,100.0,0,0
2,iec_sp_q,-22480.6,355.1,15067.7,522356,0
3,iec_sp_v,0.0,166.0,174.4,0,0
4,iec_sp_pf,-1.0,0.0,1.0,859029,0
5,ambient_temp,-270.0,24.5,48.4,2649,0
6,irradiance,-7.8,4.5,1518.3,56222,0
7,mod_temp1,-270.0,25.9,68.2,2904,0
8,mod_temp2,-270.0,25.6,67.6,2941,0
9,precipitation,-0.8,0.0,0.0,1760,0


**What it shows.** Per tag: minimum, median, maximum, and counts of negative values
and of inverter values above 1897 kW (110 % of one inverter's
share of the 3,449 kW AC capacity).

**What it means.** The raw data hold impossible values that the original pipeline left in the
target and the features (F-14):
- inverter AC power up to 40,825 kW (4 minutes
  above the flag);
- the fault code −270 °C in ambient_temp, mod_temp1, mod_temp2;
- wind speeds up to 629 m/s.

They are removed in stage 5 by documented rules, never here.

## 5. Evidence that the time base is right

In [7]:
pd.read_csv("../outputs/03_eda/tables/table_time_bug.csv")

,month,reading,clear_days,median_centroid_minus_solar_noon_min
0,"winter (Dec 2021, UTC+2)",original (local stamps read as UTC),6,120.4
1,"winter (Dec 2021, UTC+2)","rebuild (true UTC, clock-corrected)",6,1.9
2,"summer (Jul 2022, UTC+3)",original (local stamps read as UTC),6,173.6
3,"summer (Jul 2022, UTC+3)","rebuild (true UTC, clock-corrected)",6,2.6


**What it shows.** On clear days, the centre of the production curve should coincide with
solar noon. Read the original way (local stamps as UTC), it falls 120–174
minutes after solar noon (winter and summer). In the rebuild, after the true-UTC conversion and
the logger-clock correction of stage 4, it falls 1.9–2.6 minutes after.

**What it means.** Production and solar geometry are in phase. Every solar-position, clear-sky
and irradiance-based feature of the later stages depends on this.

**Next stage.** Stage 4 (`03_preprocessing.ipynb`) corrects the drift of the logger clocks and
the 2025 change of data source, and aggregates to hours.

**Reproduce.** `python -m solarpv.data.ingest`, then
`python scripts/make_data_ingest_notebook.py`.